# 第八章 · 多维成长能力

## 研报核心观点

传统成长只看收入和利润，**企业成长是多维过程**：
- **需求扩张**：卖得更多
- **盈利规模扩张**：赚得更多
- **现金流成长**：现金回流
- **经营效率提升**：单位资源产出更多
- **盈利能力**：利润质量提升

## 12 个细分因子

| 维度 | 因子 | 公式 |
|------|------|------|
| 需求扩张 | 营业收入同比 | revenue_yoy |
| 需求扩张 | 毛利润同比 | gross_profit_yoy |
| 盈利规模 | 营业利润同比 | profit_yoy |
| 盈利规模 | 净利润同比 | net_profit_yoy |
| 现金流 | CFO同比 | CFO_yoy |
| 现金流 | 自由现金流同比 | FCF_yoy |
| 现金流 | CFO/营收变化 | (CFO/Rev)_yoy |
| 经营效率 | 总资产周转率变化 | Total Assets_Turnover_yoy |
| 经营效率 | 固定资产周转率变化 | Fixed Assets_Turnover_yoy |
| 经营效率 | 存货周转率变化 | Inventory_Turnover_yoy |
| 盈利能力 | 毛利率变化 | Gross Margin_yoy |
| 盈利能力 | 净利率变化 | Net Margin_yoy |
| 盈利能力 | ROE变化 (Delta ROE) | ROE_t - ROE_{t-4} |

## 预期效果

| 维度 | 代表因子 | Rank IC | 多空年化 |
|------|----------|---------|---------|
| 传统成长 | 营收同比 | 2.68% | 11.30% |
| 传统成长 | 毛利润同比 | 3.31% | 10.23% |
| 盈利规模 | 营业利润同比 | **3.23%** | 10.54% |
| 盈利规模 | 净利润同比 | 3.18% | 10.83% |
| 现金流 | CFO同比 | 1.07% | 3.15% |
| 现金流 | CFO/营收 | 0.90% | 3.52% |
| 经营效率 | 总资产周转率 | 2.27% | 10.30% |
| 经营效率 | 存货周转率 | 1.82% | 8.11% |
| 盈利能力 | 毛利率同比 | 2.89% | 11.54% |
| 盈利能力 | ROE变化 | **3.24%** | 12.38% |
| **综合多维** | 全部 13 因子 | - | - |


## 1. 环境设置

In [ ]:
import sys
sys.path.insert(0, '.')
import 因子库

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('环境就绪')

## 2. 数据准备

In [ ]:
# 加载所有 PIT 字段
pit_revenue = 因子库.load_pit_financial('operating_revenue')
pit_gross_profit = 因子库.load_pit_financial('gross_profit')
pit_profit_op = 因子库.load_pit_financial('profit_from_operation')
pit_profit = 因子库.load_pit_financial('net_profit_parent_company')
pit_cfo = 因子库.load_pit_financial('cash_flow_from_operating_activities')
pit_assets = 因子库.load_pit_financial('total_assets')
pit_fixed = 因子库.load_pit_financial('net_fixed_assets')
pit_inventory = 因子库.load_pit_financial('inventory')
pit_equity = 因子库.load_pit_financial('equity_parent_company')

# 累计 -> 单季度
sq_revenue, _ = 因子库.to_single_quarter(pit_revenue, 'operating_revenue')
sq_gross, _ = 因子库.to_single_quarter(pit_gross_profit, 'gross_profit')
sq_profit_op, _ = 因子库.to_single_quarter(pit_profit_op, 'profit_from_operation')
sq_profit, _ = 因子库.to_single_quarter(pit_profit, 'net_profit_parent_company')
sq_cfo, _ = 因子库.to_single_quarter(pit_cfo, 'cash_flow_from_operating_activities')
sq_assets, _ = 因子库.to_single_quarter(pit_assets, 'total_assets')
sq_fixed, _ = 因子库.to_single_quarter(pit_fixed, 'net_fixed_assets')
sq_inventory, _ = 因子库.to_single_quarter(pit_inventory, 'inventory')
sq_equity, _ = 因子库.to_single_quarter(pit_equity, 'equity_parent_company')

print(f'营收: {sq_revenue.shape}')
print(f'毛利润: {sq_gross.shape}')
print(f'净利润: {sq_profit.shape}')
print(f'CFO: {sq_cfo.shape}')
print(f'总资产: {sq_assets.shape}')

# 行情数据
close_price = 因子库.load_daily('close')
returns_next_month = 因子库.next_month_return(close_price, months=1)

## 3. 维度 1：需求扩张因子

**核心**：营业收入同比、毛利润同比

In [ ]:
yoy_revenue = sq_revenue / sq_revenue.shift(4, axis=1) - 1
yoy_gross_profit = sq_gross / sq_gross.shift(4, axis=1) - 1

for name, factor in [('营收同比', yoy_revenue), ('毛利润同比', yoy_gross_profit)]:
    ic = 因子库.rank_ic_test(factor, returns_next_month)
    stats = 因子库.calc_ic_stats(ic)
    print(f'{name}: IC={stats["mean"]:.4f}, ICIR={stats["icir"]:.4f}')

## 4. 维度 2：盈利规模扩张因子

**核心**：营业利润同比、净利润同比

In [ ]:
yoy_profit_op = sq_profit_op / sq_profit_op.shift(4, axis=1) - 1
yoy_profit = sq_profit / sq_profit.shift(4, axis=1) - 1

for name, factor in [('营业利润同比', yoy_profit_op), ('净利润同比', yoy_profit)]:
    ic = 因子库.rank_ic_test(factor, returns_next_month)
    stats = 因子库.calc_ic_stats(ic)
    print(f'{name}: IC={stats["mean"]:.4f}, ICIR={stats["icir"]:.4f}')

## 5. 维度 3：现金流成长因子

**核心**：CFO 同比、CFO/营收 变化

In [ ]:
yoy_cfo = sq_cfo / sq_cfo.shift(4, axis=1) - 1

cfo_to_rev = sq_cfo / sq_revenue.replace(0, np.nan)
yoy_cfo_to_rev = cfo_to_rev / cfo_to_rev.shift(4, axis=1) - 1

print('现金流因子:')
for name, factor in [('CFO同比', yoy_cfo), ('CFO/营收变化', yoy_cfo_to_rev)]:
    ic = 因子库.rank_ic_test(factor, returns_next_month)
    stats = 因子库.calc_ic_stats(ic)
    print(f'  {name}: IC={stats["mean"]:.4f}, ICIR={stats["icir"]:.4f}')

## 6. 维度 4：经营效率提升因子

**核心**：总资产周转率、固定资产周转率、存货周转率

In [ ]:
asset_turnover = sq_revenue / sq_assets.replace(0, np.nan)
yoy_asset_turnover = asset_turnover / asset_turnover.shift(4, axis=1) - 1

fixed_turnover = sq_revenue / sq_fixed.replace(0, np.nan)
yoy_fixed_turnover = fixed_turnover / fixed_turnover.shift(4, axis=1) - 1

inv_turnover = sq_revenue / sq_inventory.replace(0, np.nan)
yoy_inv_turnover = inv_turnover / inv_turnover.shift(4, axis=1) - 1

print('经营效率因子:')
for name, factor in [
    ('总资产周转率变化', yoy_asset_turnover),
    ('固定资产周转率变化', yoy_fixed_turnover),
    ('存货周转率变化', yoy_inv_turnover)
]:
    ic = 因子库.rank_ic_test(factor, returns_next_month)
    stats = 因子库.calc_ic_stats(ic)
    print(f'  {name}: IC={stats["mean"]:.4f}, ICIR={stats["icir"]:.4f}')

## 7. 维度 5：盈利能力与定价权因子

**核心**：毛利率、净利率、ROE（Delta ROE）

In [ ]:
gross_margin = sq_gross / sq_revenue.replace(0, np.nan)
yoy_gross_margin = gross_margin - gross_margin.shift(4, axis=1)

net_margin = sq_profit / sq_revenue.replace(0, np.nan)
yoy_net_margin = net_margin - net_margin.shift(4, axis=1)

roe = sq_profit / sq_equity.replace(0, np.nan)
delta_roe = roe - roe.shift(4, axis=1)

print('盈利能力因子:')
for name, factor in [
    ('毛利率变化', yoy_gross_margin),
    ('净利率变化', yoy_net_margin),
    ('Delta ROE', delta_roe)
]:
    ic = 因子库.rank_ic_test(factor, returns_next_month)
    stats = 因子库.calc_ic_stats(ic)
    print(f'  {name}: IC={stats["mean"]:.4f}, ICIR={stats["icir"]:.4f}')

## 8. 多维度综合因子

把 5 个维度的因子**标准化后等权平均**

In [ ]:
dim1_demand = 因子库.normalize_factor(yoy_revenue)
dim2_profit = 因子库.normalize_factor(yoy_profit)
dim3_cashflow = 因子库.normalize_factor(yoy_cfo)
dim4_efficiency = 因子库.normalize_factor(yoy_asset_turnover)
dim5_profitability = 因子库.normalize_factor(delta_roe)

common = dim1_demand.index
common_stocks = dim1_demand.columns

factor_multidim = (
    dim1_demand +
    dim2_profit +
    dim3_cashflow +
    dim4_efficiency +
    dim5_profitability
) / 5
print(f'多维成长综合因子: {factor_multidim.shape}')

ic_md = 因子库.rank_ic_test(factor_multidim, returns_next_month)
stats_md = 因子库.calc_ic_stats(ic_md)
print(f'多维成长综合: IC={stats_md["mean"]:.4f}, ICIR={stats_md["icir"]:.4f}')

decile_md = 因子库.decile_test(factor_multidim, returns_next_month)
if decile_md:
    long_short = decile_md[max(decile_md.keys())] - decile_md[min(decile_md.keys())]
    annualized = (1 + long_short) ** 12 - 1
    print(f'多空年化: {annualized:.4f}')

fig = 因子库.plot_ic_curve(ic_md, title='多维成长综合 - Rank IC')
plt.show()
fig = 因子库.plot_decile_bars(decile_md, title='多维成长综合 - 十分组收益')
plt.show()

## 9. 因子汇总

| 因子 | 维度 | 我们的 IC | 研报 IC |
|------|------|----------|---------|
| 营收同比 | 需求 | 见上 | 2.68% |
| 毛利润同比 | 需求 | 见上 | 3.31% |
| 营业利润同比 | 盈利 | 见上 | **3.23%** |
| 净利润同比 | 盈利 | 见上 | 3.18% |
| CFO同比 | 现金流 | 见上 | 1.07% |
| CFO/营收 | 现金流 | 见上 | 0.90% |
| 总资产周转率 | 效率 | 见上 | 2.27% |
| 存货周转率 | 效率 | 见上 | 1.82% |
| 毛利率变化 | 盈利 | 见上 | 2.89% |
| Delta ROE | 盈利 | 见上 | **3.24%** |
| **多维成长综合** | - | 见上 | - |

### 下一步
- **第九章**：低基数效应（条件剔除）
- **第十章**：综合成长补充因子（融合前几章）